In [ ]:
import geopandas as gpd
import config

UKRAINE_ALBERS_WKT = config.UKRAINE_ALBERS_WKT 

quads_path = config.GEOAI_ROOT / "download_basemaps" / "all_quads.geojson"
if not quads_path.exists():
    quads_path = config.GEOAI_ROOT / "delineation" / "all_quads.geojson"
quads = gpd.read_file(quads_path)
if quads.crs is None:
    quads = quads.set_crs("EPSG:4326")

quads_alb = quads.to_crs(UKRAINE_ALBERS_WKT)

In [2]:
import numpy as np
import rasterio
from rasterio.features import geometry_window, geometry_mask

def compute_cell_stats(
    quads_alb: gpd.GeoDataFrame,
    raster_2021: str,
    raster_2025: str,
    raster_loss: str,
) -> gpd.GeoDataFrame:
    """Add area metrics per quad:
       - cell_area_km2 (6 decimals)
       - belt_2021_ha, belt_2025_ha, belt_loss_ha (4 decimals)
       - belt_2021_ha_per_km2, belt_2025_ha_per_km2, belt_loss_ha_per_km2 (4 decimals)
    """
    # Open all rasters
    with rasterio.open(raster_2021) as r21, \
         rasterio.open(raster_2025) as r25, \
         rasterio.open(raster_loss) as rL:

        # sanity: all rasters same grid
        assert r21.transform == r25.transform == rL.transform
        assert r21.width == r25.width == rL.width
        assert r21.height == r25.height == rL.height

        # pixel area in m² (Albers equal area)
        pixel_area_m2 = abs(r21.transform.a * r21.transform.e)

        # Prepare result columns
        cell_area_km2   = []
        belt_2021_ha    = []
        belt_2025_ha    = []
        belt_loss_ha    = []
        belt_2021_dens  = []
        belt_2025_dens  = []
        belt_loss_dens  = []

        for geom in quads_alb.geometry:
            if geom.is_empty:
                cell_area_km2.append(0.0)
                belt_2021_ha.append(0.0)
                belt_2025_ha.append(0.0)
                belt_loss_ha.append(0.0)
                belt_2021_dens.append(0.0)
                belt_2025_dens.append(0.0)
                belt_loss_dens.append(0.0)
                continue

            # 1) Cell area in km² (geometry in metres)
            area_km2 = float(geom.area) / 1_000_000.0

            # 2) Get window covering this quad
            win = geometry_window(r21, [geom], pad_x=0, pad_y=0, north_up=True)

            # Read strips from rasters for this window
            a21 = r21.read(1, window=win)
            a25 = r25.read(1, window=win)
            aL  = rL.read(1, window=win)

            # Mask outside geometry within the window
            mask = geometry_mask(
                [geom],
                transform=r21.window_transform(win),
                invert=True,
                out_shape=a21.shape,
            )

            # count belt pixels inside geometry
            n21 = np.logical_and(a21 == 1, mask).sum()
            n25 = np.logical_and(a25 == 1, mask).sum()
            nL  = np.logical_and(aL  == 1, mask).sum()

            # convert to ha (1 ha = 10,000 m²)
            ha21 = n21 * pixel_area_m2 / 10_000.0
            ha25 = n25 * pixel_area_m2 / 10_000.0
            haL  = nL * pixel_area_m2 / 10_000.0

            # densities: X ha per 1 km²
            if area_km2 > 0:
                d21 = ha21 / area_km2
                d25 = ha25 / area_km2
                dL  = haL  / area_km2
            else:
                d21 = d25 = dL = 0.0

            # rounding
            cell_area_km2.append(round(area_km2, 6))
            belt_2021_ha.append(round(ha21, 4))
            belt_2025_ha.append(round(ha25, 4))
            belt_loss_ha.append(round(haL, 4))
            belt_2021_dens.append(round(d21, 4))
            belt_2025_dens.append(round(d25, 4))
            belt_loss_dens.append(round(dL, 4))

    # Attach columns
    quads_out = quads_alb.copy()
    quads_out["cell_area_km2"]           = cell_area_km2
    quads_out["belt_2021_ha"]           = belt_2021_ha
    quads_out["belt_2025_ha"]           = belt_2025_ha
    quads_out["belt_loss_2021_2025_ha"] = belt_loss_ha
    quads_out["belt_2021_ha_per_km2"]   = belt_2021_dens
    quads_out["belt_2025_ha_per_km2"]   = belt_2025_dens
    quads_out["loss_ha_per_km2"]        = belt_loss_dens

    return quads_out

In [ ]:
quads_stats = compute_cell_stats(
    quads_alb=quads_alb,
    raster_2021=str(config.GEOAI_ROOT / "outputs" / "2021" / ("global_monthly_2021_voted_cl123_albers.vrt" if (config.GEOAI_ROOT / "outputs" / "2021" / "global_monthly_2021_voted_cl123_albers.vrt").exists() else "global_monthly_2021_voted_cl123_albers.tif")),
    raster_2025=str(config.GEOAI_ROOT / "outputs" / "2025" / ("global_monthly_2025_clean_cl123_albers.vrt" if (config.GEOAI_ROOT / "outputs" / "2025" / "global_monthly_2025_clean_cl123_albers.vrt").exists() else "global_monthly_2025_clean_cl123_albers.tif")),
    raster_loss=str(config.GEOAI_ROOT / "outputs" / "2025" / "global_monthly_2021_2025_loss_cl123_albers.tif" if (config.GEOAI_ROOT / "outputs" / "2025" / "global_monthly_2021_2025_loss_cl123_albers.tif").exists() else config.GEOAI_ROOT / "outputs" / "global_monthly_2021_2025_loss_cl123_albers.tif"),
)

quads_stats.to_file(str(config.GEOAI_ROOT / "outputs" / "all_quads_belt_stats.gpkg"), driver="GPKG")

In [ ]:
# --- Accuracy assessment: configuration + verified Olofsson (2014) estimator ---
from pathlib import Path

import numpy as np
import rasterio
from rasterio.features import rasterize, geometry_window, geometry_mask
import geopandas as gpd

import config

# ---------------------------------------------------------------------------
# CONFIG  (edit paths here).  Defaults use the Google-Drive (G:) layout.
# ---------------------------------------------------------------------------
GEOAI = config.GEOAI_ROOT

PRED_RASTER = {
    "2021": (GEOAI / "outputs" / "2021" / "global_monthly_2021_voted_cl123_albers.vrt" if (GEOAI / "outputs" / "2021" / "global_monthly_2021_voted_cl123_albers.vrt").exists() else GEOAI / "outputs" / "2021" / "global_monthly_2021_voted_cl123_albers.tif"),
    "2025": (GEOAI / "outputs" / "2025" / "global_monthly_2025_clean_cl123_albers.vrt" if (GEOAI / "outputs" / "2025" / "global_monthly_2025_clean_cl123_albers.vrt").exists() else GEOAI / "outputs" / "2025" / "global_monthly_2025_clean_cl123_albers.tif"),
}
RASTER_LOSS = GEOAI / "outputs" / "global_monthly_2021_2025_loss_cl123_albers.tif"
REF_DIR = {
    "2021": GEOAI / "masks" / "2021" / "test",
    "2025": GEOAI / "masks" / "2025" / "test",
}
quads_candidates = [GEOAI / "download_basemaps" / "all_quads.geojson", GEOAI / "delineation" / "all_quads.geojson"]
QUADS_PATH = next((p for p in quads_candidates if p.exists()), quads_candidates[0])
TEST_QUADS = config.load_tile_splits()["test"]      # ['1222-1323', '1228-1353', '1240-1341']

ALBERS = config.UKRAINE_ALBERS_WKT
BELT, BACK = 0, 1          # class order in the confusion matrix: 0 = belt, 1 = background
M2_PER_HA = 10_000.0


def olofsson_area_estimate(counts, map_area, z=1.96):
    """Stratified error-adjusted area & accuracy estimator
    (Olofsson et al. 2014, Remote Sensing of Environment 148:42-57).

    Parameters
    ----------
    counts : (K, K) array of ints
        Error matrix of SAMPLE COUNTS n_ij.  MAP class = ROWS (i), REFERENCE class = COLUMNS (j).
    map_area : (K,) array
        Mapped AREA of each map class over the population (any unit); A_tot = sum(map_area).
    z : float
        Normal quantile for the CI (1.96 = 95%).

    Returns a dict with W, A_tot, p, p_area, A_adj, SE_A, MoE_A (area + 95% half-width),
    U/SE_U/MoE_U (user's acc), P/SE_P/MoE_P (producer's acc / recall), O/SE_O/MoE_O (overall).

    Equation map: p_ij = W_i n_ij/n_i.  Eq.(4);  A_hat_k = A_tot * sum_i W_i n_ik/n_i.  Eq.(9);
    S(p_.k) = sqrt(sum_i W_i^2 r_ik(1-r_ik)/(n_i.-1))  Eq.(10);  S(A_hat) = A_tot*S(p_.k)  Eq.(11).
    Validated to reproduce the paper's Table 8 worked example exactly (see self-test cell).
    """
    n = np.asarray(counts, dtype=float)
    K = n.shape[0]
    if n.shape != (K, K):
        raise ValueError("counts must be a square KxK matrix")

    map_area = np.asarray(map_area, dtype=float)
    A_tot = map_area.sum()
    W = map_area / A_tot                       # stratum weights, sum -> 1

    ni = n.sum(axis=1)                         # row totals  n_i.
    nonempty = ni > 0

    # estimated area proportions  p_ij = W_i * n_ij / n_i.
    rr = np.divide(n, ni[:, None], out=np.zeros_like(n), where=ni[:, None] > 0)
    p = W[:, None] * rr
    p_area = p.sum(axis=0)                      # p_.k  (column sums)
    A_adj = A_tot * p_area

    # SE of the area proportion (Eq. 10): n_i.=1 -> term 0; empty row -> 0
    denom = ni - 1.0
    safe = nonempty & (ni > 1)
    term = np.zeros((K, K))
    contrib = (W[:, None] ** 2) * rr * (1.0 - rr)
    term[safe, :] = contrib[safe, :] / denom[safe, None]
    SE_p_area = np.sqrt(term.sum(axis=0))
    SE_A = A_tot * SE_p_area
    MoE_A = z * SE_A

    # user's accuracy  U_i = n_ii / n_i.
    diag = np.diag(n)
    U = np.divide(diag, ni, out=np.full(K, np.nan), where=ni > 0)
    SE_U = np.full(K, np.nan)
    SE_U[safe] = np.sqrt(U[safe] * (1.0 - U[safe]) / denom[safe])
    MoE_U = z * SE_U

    # overall accuracy  O = sum_i W_i n_ii/n_i.
    Uw = np.where(nonempty, U, 0.0)
    O = np.sum(W * Uw)
    var_O = np.zeros(K)
    var_O[safe] = (W[safe] ** 2) * U[safe] * (1.0 - U[safe]) / denom[safe]
    SE_O = np.sqrt(var_O.sum())
    MoE_O = z * SE_O

    # producer's accuracy  P_j = p_jj / p_.j  (Olofsson Eq. 7 variance)
    pjj = np.diag(p)
    P = np.divide(pjj, p_area, out=np.full(K, np.nan), where=p_area > 0)
    Ni = W * A_tot
    Nj_hat = np.array([np.sum(np.where(nonempty, Ni / ni, 0.0) * n[:, j]) for j in range(K)])
    SE_P = np.full(K, np.nan)
    for j in range(K):
        if not (p_area[j] > 0 and Nj_hat[j] > 0):
            continue
        Uj = U[j] if nonempty[j] else 0.0
        t1 = (Ni[j] ** 2) * (1.0 - P[j]) ** 2 * Uj * (1.0 - Uj) / denom[j] if safe[j] else 0.0
        t2 = 0.0
        for i in range(K):
            if i == j or not safe[i]:
                continue
            rij = n[i, j] / ni[i]
            t2 += (Ni[i] ** 2) * rij * (1.0 - rij) / denom[i]
        t2 *= P[j] ** 2
        SE_P[j] = np.sqrt((t1 + t2) / Nj_hat[j] ** 2)
    MoE_P = z * SE_P

    return dict(W=W, A_tot=A_tot, p=p, p_area=p_area, A_adj=A_adj,
                SE_p_area=SE_p_area, SE_A=SE_A, MoE_A=MoE_A,
                U=U, SE_U=SE_U, MoE_U=MoE_U, P=P, SE_P=SE_P, MoE_P=MoE_P,
                O=O, SE_O=SE_O, MoE_O=MoE_O)


In [ ]:
# --- Self-test: reproduce Olofsson et al. (2014) Table 8 worked example ---
# If this passes, the estimator matches the published numbers (deforestation 21,158 +/- 6,158 ha).
_counts = np.array([[66, 0, 5, 4], [0, 55, 8, 12], [1, 0, 153, 11], [2, 1, 9, 313]], float)
_w = np.array([200_000, 150_000, 3_200_000, 6_450_000], float)   # mapped pixels per class
_r = olofsson_area_estimate(_counts, _w)
_PIX2HA = 0.09
_area_ha = _r["A_adj"] * _PIX2HA
_moe_ha = _r["MoE_A"] * _PIX2HA
assert np.allclose(_area_ha, [21158, 11686, 285770, 581386], atol=2), _area_ha
assert np.allclose(_moe_ha, [6158, 3756, 15510, 16282], atol=2), _moe_ha
assert round(_r["O"], 2) == 0.95, _r["O"]          # published overall accuracy
print("Olofsson estimator self-test PASSED  "
      "(deforestation 21,158 +/- 6,158 ha; OA 0.95 — matches the published worked example).")


In [ ]:
# --- Helpers: load quads, build the pixel confusion matrix, and population weights ---

def _ensure_crs(gdf):
    """Attach a CRS if the file lacks one (infer 4326 vs 3857 from coordinate magnitude)."""
    if gdf.crs is not None:
        return gdf
    minx, miny, maxx, maxy = gdf.total_bounds
    big = max(abs(minx), abs(maxx), abs(miny), abs(maxy)) > 360
    return gdf.set_crs("EPSG:3857" if big else "EPSG:4326")


def load_quads(quads_path, test_quads):
    """Return ({quad_id: geom in Albers} for the test quads, full quad GeoDataFrame, id column)."""
    g = _ensure_crs(gpd.read_file(quads_path)).to_crs(ALBERS)
    id_col = next((c for c in g.columns
                   if c != "geometry" and g[c].astype(str).isin(test_quads).any()), None)
    if id_col is None:
        raise ValueError(f"No column in {quads_path} matches quad ids {test_quads}")
    test_geoms = {str(r[id_col]): r.geometry for _, r in g.iterrows()
                  if str(r[id_col]) in test_quads}
    missing = set(test_quads) - set(test_geoms)
    if missing:
        raise ValueError(f"Test quads not found in quad layer: {missing}")
    return test_geoms, g, id_col


def _quad_arrays(pred_src, ref_geojson, quad_geom):
    """For one quad window return boolean (map_belt, ref_belt, inside) on the prediction grid."""
    win = geometry_window(pred_src, [quad_geom])
    transform = pred_src.window_transform(win)
    pred = pred_src.read(1, window=win)
    shape = pred.shape

    ref = _ensure_crs(gpd.read_file(ref_geojson)).to_crs(ALBERS)
    geoms = [gm for gm in ref.geometry if gm is not None and not gm.is_empty]
    ref_belt = (rasterize([(gm, 1) for gm in geoms], out_shape=shape, transform=transform,
                          fill=0, dtype=np.uint8, all_touched=False).astype(bool)
                if geoms else np.zeros(shape, bool))
    inside = geometry_mask([quad_geom], transform=transform, invert=True, out_shape=shape)
    return (pred == 1) & inside, ref_belt & inside, inside


def build_confusion(year, quad_geoms, return_blocks=False, block_px=256):
    """2x2 confusion (rows = map [belt, back], cols = ref [belt, back]) over the test quads.

    Predicted belt = pixel value 1; predicted background = any other value inside the quad
    (the voted/clean mosaics store belt=1, 255=nodata). Optionally also returns a list of
    per-block 2x2 matrices (square blocks of `block_px`) for the spatial block bootstrap.
    """
    counts = np.zeros((2, 2), np.int64)
    blocks = []
    with rasterio.open(PRED_RASTER[year]) as src:
        for qid, geom in quad_geoms.items():
            ref_geojson = REF_DIR[year] / f"{qid}.geojson"
            if not ref_geojson.exists():
                raise FileNotFoundError(ref_geojson)
            map_belt, ref_belt, inside = _quad_arrays(src, ref_geojson, geom)
            map_back = inside & ~map_belt

            def cm(mb, rb, bk):
                return np.array([[np.sum(mb & rb), np.sum(mb & ~rb)],
                                 [np.sum(bk & rb), np.sum(bk & ~rb)]], np.int64)

            counts += cm(map_belt, ref_belt, map_back)
            if return_blocks:
                H, W = inside.shape
                for y0 in range(0, H, block_px):
                    for x0 in range(0, W, block_px):
                        s = (slice(y0, y0 + block_px), slice(x0, x0 + block_px))
                        if inside[s].any():
                            blocks.append(cm(map_belt[s], ref_belt[s], map_back[s]))
    return (counts, blocks) if return_blocks else counts


def population_weights(year, all_quads_gdf):
    """Mapped belt & background AREA (m^2) over ALL quads = the strata of the population."""
    n_belt = n_tot = 0
    with rasterio.open(PRED_RASTER[year]) as src:
        pix_area = abs(src.transform.a * src.transform.e)
        for geom in all_quads_gdf.geometry:
            if geom is None or geom.is_empty:
                continue
            win = geometry_window(src, [geom])
            transform = src.window_transform(win)
            arr = src.read(1, window=win)
            inside = geometry_mask([geom], transform=transform, invert=True, out_shape=arr.shape)
            n_tot += int(inside.sum())
            n_belt += int(((arr == 1) & inside).sum())
    a_belt = n_belt * pix_area
    return np.array([a_belt, n_tot * pix_area - a_belt]), pix_area   # [belt, background]


In [ ]:
# --- Run the accuracy assessment for 2021 and 2025 ---
quad_geoms, all_quads_gdf, _id_col = load_quads(QUADS_PATH, TEST_QUADS)
print(f"Test quads ({_id_col}): {list(quad_geoms)}   |   population quads: {len(all_quads_gdf)}\n")

results = {}
for year in ["2021", "2025"]:
    counts = build_confusion(year, quad_geoms)
    map_area, pix_area = population_weights(year, all_quads_gdf)
    est = olofsson_area_estimate(counts, map_area)
    results[year] = dict(counts=counts, map_area=map_area, pix_area=pix_area, est=est)

    tp, fp = counts[BELT]
    fn, tn = counts[BACK]
    print(f"========================  {year}  ========================")
    print(f"Confusion matrix  (rows = map [belt, background], cols = reference [belt, background]):")
    print(f"   {counts}")
    print(f"   TP={tp:,}  FP={fp:,}  FN={fn:,}  TN={tn:,}   (N = {counts.sum():,} px)")
    print(f"   pixel size: {pix_area:.2f} m^2")
    print(f"Mapped belt area        : {map_area[BELT] / M2_PER_HA:12,.0f} ha "
          f"(W_belt = {est['W'][BELT]:.4f})")
    print(f"Error-adjusted belt area: {est['A_adj'][BELT] / M2_PER_HA:12,.0f} "
          f"+/- {est['MoE_A'][BELT] / M2_PER_HA:,.0f} ha   (95% CI, per-pixel Olofsson Eq. 9-11)")
    print(f"Overall accuracy        : {est['O']:.3f} +/- {est['MoE_O']:.3f}")
    print(f"Belt   user's accuracy  : {est['U'][BELT]:.3f} +/- {est['MoE_U'][BELT]:.3f}")
    print(f"Belt   producer's (recall): {est['P'][BELT]:.3f} +/- {est['MoE_P'][BELT]:.3f}")
    print(f"Backgr producer's (recall): {est['P'][BACK]:.3f} +/- {est['MoE_P'][BACK]:.3f}\n")


In [ ]:
# --- Cluster-aware uncertainty (spatial block bootstrap) + error-adjusted net loss ---
# The per-pixel Olofsson CI is optimistic because test pixels are spatially autocorrelated.
# We resample 256-px (~0.77 km) blocks with replacement and recompute the adjusted area each
# time; the 2.5/97.5 percentiles are the HEADLINE 95% CI.
rng = np.random.default_rng(config.SEED)
N_BOOT = 1000
BLOCK_PX = 256


def bootstrap_adjusted_belt(year, quad_geoms, map_area, n_boot, block_px, rng):
    """Block-bootstrap distribution of the error-adjusted belt area (m^2)."""
    counts, blocks = build_confusion(year, quad_geoms, return_blocks=True, block_px=block_px)
    blocks = np.stack(blocks)                       # (B, 2, 2)
    B = len(blocks)
    draws = np.empty(n_boot)
    for b in range(n_boot):
        cm = blocks[rng.integers(0, B, size=B)].sum(axis=0)
        draws[b] = olofsson_area_estimate(cm, map_area)["A_adj"][BELT]
    point = olofsson_area_estimate(counts, map_area)["A_adj"][BELT]
    return point, draws, B


boot = {}
for year in ["2021", "2025"]:
    point, draws, B = bootstrap_adjusted_belt(
        year, quad_geoms, results[year]["map_area"], N_BOOT, BLOCK_PX, rng)
    lo, hi = np.percentile(draws, [2.5, 97.5])
    boot[year] = dict(point=point, draws=draws, lo=lo, hi=hi, B=B)
    print(f"{year}: adjusted belt {point / M2_PER_HA:11,.0f} ha   "
          f"block-bootstrap 95% CI [{lo / M2_PER_HA:,.0f}, {hi / M2_PER_HA:,.0f}] ha "
          f"(B={B} blocks)")

# --- Net war-driven loss = adjusted(2021) - adjusted(2025) -------------------
A21, SE21 = results["2021"]["est"]["A_adj"][BELT], results["2021"]["est"]["SE_A"][BELT]
A25, SE25 = results["2025"]["est"]["A_adj"][BELT], results["2025"]["est"]["SE_A"][BELT]
loss = A21 - A25

# (a) Per-pixel Olofsson propagation: independent annual estimates -> variances add (Eq. 11).
se_loss = np.sqrt(SE21 ** 2 + SE25 ** 2)
# (b) Block-bootstrap loss CI: independent resamples per year -> conservative if the same
#     quads validate both years (positive covariance ignored => interval slightly wide).
loss_draws = boot["2021"]["draws"] - boot["2025"]["draws"]
llo, lhi = np.percentile(loss_draws, [2.5, 97.5])

print("\n================  ERROR-ADJUSTED NET LOSS  ================")
print(f"Net loss point estimate : {loss / M2_PER_HA:12,.0f} ha "
      f"({100 * loss / A21:.1f}% of the 2021 adjusted belt area)")
print(f"  per-pixel 95% CI      : +/- {1.96 * se_loss / M2_PER_HA:,.0f} ha "
      f"(optimistic lower bound on uncertainty)")
print(f"  block-bootstrap 95% CI: [{llo / M2_PER_HA:,.0f}, {lhi / M2_PER_HA:,.0f}] ha  "
      f"<-- headline")
print(f"  significant decline (bootstrap CI excludes 0): {bool(llo > 0)}")

# --- Cross-check against the raw mapped loss raster (cell above) -------------
try:
    with rasterio.open(RASTER_LOSS) as src:
        pa = abs(src.transform.a * src.transform.e)
        nloss = sum(int((src.read(1, window=geometry_window(src, [g])) == 1).sum())
                    for g in all_quads_gdf.geometry if g is not None and not g.is_empty)
    print(f"\nRaw mapped loss raster   : {nloss * pa / M2_PER_HA:,.0f} ha "
          f"(unadjusted; for comparison only)")
except Exception as e:
    print(f"\n[raw loss raster not read: {e}]")
